# 7.4 Text Mining in Higher Ed – Topic Modeling



## Introduction
In the previous module, we learned how to transform raw student comments into mathematical vectors. However, once we have thousands of these vectors, the next challenge is making sense of them at scale. This is where **Topic Modeling** becomes a vital tool for Institutional Research.

Topic modeling is an **unsupervised machine learning** technique. Unlike the classification models we built in Course 2, topic modeling does not require pre-labeled data. Instead, it "discovers" recurring themes and patterns across a massive corpus of text, allowing us to categorize student voices without the bias or manual labor of reading every individual response.

**In this module, we will explore:**
* The strategic value of **Unsupervised Learning** in a Higher Ed context.
* The mechanics of two primary algorithms: **NMF** (Non-negative Matrix Factorization) and **LDA** (Latent Dirichlet Allocation).
* How to interpret **Top-Weighted Words** to define institutional themes.
* The workflow for **Assigning Dominant Topics** to specific students to create new categorical features.
* How to translate algorithmic outputs into **Stakeholder-Ready Reports** that drive institutional action.

<br>

#### **Guiding Questions**
1. If we have thousands of open-ended comments, how can we discover the main themes without reading every response?
2. How do we interpret and communicate topics to institutional stakeholders?

<br>

#### **Learning Objectives**
By the end of this notebook, you will be able to:
* **Explain** the purpose and mechanics of topic modeling.
* **Apply NMF** (with TF-IDF) and **LDA** (with count vectors) using scikit-learn.
* **Assign** a dominant topic to each student response.
* **Produce** a simple, IR-ready summary of results.

> **Instructor Perspective:** Topic modeling turns "unstructured noise" into "structured insight." As an analyst, your role is to act as the interpreter—using these mathematical clusters to tell a coherent story about the student experience to your campus leadership.

<br>


## 1. Why Topic Modeling in IR?



Institutional Research (IR) teams often collect massive amounts of free-text data—including course evaluations, exit surveys, and advising feedback forms. Reading and coding these manually is not only labor-intensive but introduces subjective human bias.

**Topic modeling** is an **unsupervised** technique that addresses this by identifying recurring themes without requiring you to pre-define categories. By using mathematical patterns, we can let the "student voice" dictate what the most important themes are, rather than forcing their responses into our own preconceived buckets.

In this notebook, we will implement the two most common algorithms used in the field:

| Algorithm | Input Requirement | Best Institutional Use Case |
| :--- | :--- | :--- |
| **NMF** (Non-negative Matrix Factorization) | **TF-IDF** Vectors | Generating clean, highly interpretable topics for short survey comments. |
| **LDA** (Latent Dirichlet Allocation) | **Count (BoW)** Vectors | Identifying probabilistic mixtures of words in longer, more complex documents. |

<br>

> **Key Concept:** Both algorithms produce the same essential output: a list of **top-weighted words** that define each topic and a **topic weight** assigned to every individual student response. This allows us to turn qualitative text into quantitative, analyzable features.


## 2. Setup and Data Preparation


In this section, we mount our Google Drive and load the survey dataset containing the open-ended student responses for analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import random, os
import pandas as pd

In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data19 = pd.read_csv(f'{filepath}ML_Survey_Data19.csv')
display(ML_Survey_Data19)
#ML_Survey_Data19

,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,Free_Response_Text
0,UHKR1SFYP,2,3,2,2,2,2,2,2,3,2,"When multiple deadlines hit at once, I have to..."
1,UHNF02H9O,2,2,1,1,1,2,1,1,1,1,"When deadlines pile up, I feel like I’m just t..."
2,UHOFN46WE,3,3,4,3,4,3,3,3,3,3,"My academic journey has been very rewarding, a..."
3,UHP7B9BTV,2,1,2,1,1,2,1,1,2,1,Sometimes I don’t understand what instructors ...
4,UI30HVLIL,2,2,2,2,2,2,2,2,2,2,"I’ve been feeling overwhelmed, and I’m trying ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
5157,N26B5Y6HE,2,2,2,1,2,2,2,2,2,2,"I know support exists, but it can be hard to r..."
5158,N28LHIN72,2,3,3,2,3,3,2,3,3,3,"The workload can feel heavy at times, but I ca..."
5159,N299IIGIN,4,4,4,4,4,4,3,4,4,3,I am confident in my study habits and ability ...
5160,N2C5YLPI1,3,3,3,2,3,3,3,2,3,3,I am adapting to college life and finding my r...


## 3. TF-IDF Vectors (Recap)




Both NMF and LDA require a numeric document-term matrix as input. In this section, we transform our preprocessed text into a **TF-IDF (Term Frequency-Inverse Document Frequency)** matrix.

This specific vectorization method is ideal for **NMF** because it down-weights common words and highlights terms that are more unique and descriptive of specific student concerns. We will store these features to help interpret our topics later.


In [ ]:
tfidf_vec = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf_vec.fit_transform(ML_Survey_Data19['Free_Response_Text'])
feature_names_tfidf = tfidf_vec.get_feature_names_out()

print("TF-IDF matrix:", tfidf_matrix.shape, "  (responses × terms)")


TF-IDF matrix: (5162, 612)   (responses × terms)



## 4. NMF Topic Modeling


**Non-negative Matrix Factorization (NMF)** is a linear algebraic approach to topic modeling. It is highly effective for shorter documents, such as survey comments, because it tends to produce very distinct and "crisp" topics.

Conceptually, NMF factorizes our high-dimensional document-term matrix into two smaller, more manageable matrices:
* **W (The Document-Topic Matrix):** This tells us how much each individual student response belongs to each discovered topic.
* **H (The Topic-Term Matrix):** This tells us which words define each topic.

In the code below, we define the number of topics to discover (starting with a small number like 4) and then extract the **top-weighted words** from the **H matrix** to identify the underlying themes.

<br>

> **Instructor Perspective:** Think of NMF as a "feature extraction" tool. It takes hundreds of word-columns and compresses them into just a few "topic-columns" that represent the major pillars of the student experience.

In [ ]:
N_TOPICS = 4  # start small; adjust after seeing results

nmf_model = NMF(n_components=N_TOPICS, random_state=42)
W_nmf = nmf_model.fit_transform(tfidf_matrix)   # document-topic
H_nmf = nmf_model.components_                   # topic-term

def print_top_words(H, feature_names, n_words=8, label='Topic'):
    for i, row in enumerate(H):
        top_idx = row.argsort()[:-n_words-1:-1]
        words = [feature_names[j] for j in top_idx]
        print(f"  {label} {i}: {', '.join(words)}")

print("NMF Topics (top 8 words each):")
print_top_words(H_nmf, feature_names_tfidf)


NMF Topics (top 8 words each):
  Topic 0: learning, meet, learning meet, expected learning, expected, classes, challenging expected, meet expectations
  Topic 1: assignments early, feel heavy, heavy times, heavy, times manage, manage start, workload feel, start assignments
  Topic 2: helps, best, feedback, times, improve, sure, study, apply
  Topic 3: depends, organized, work helpful, team, group, depends organized, group work, helpful depends


We will rely on Gemini to generate topic labels based on the four lists above.

In [ ]:
# Prompt: Identify topics that summarize each of the sets of four words here: (Paste NMF Topic groups here)
topic_labels_nmf = {
    0: "Academic Expectations & Learning Experience",
    1: "Workload Management & Stress",
    2: "Support, Feedback & Improvement Strategies",
    3: "Group Work & Collaboration Dynamics"
}
print("Summarized NMF topic labels:")
for t_id, label in topic_labels_nmf.items():
    print(f"Topic {t_id}: {label}")

Summarized NMF topic labels:
Topic 0: Academic Expectations & Learning Experience
Topic 1: Workload Management & Stress
Topic 2: Support, Feedback & Improvement Strategies
Topic 3: Group Work & Collaboration Dynamics



## 5. LDA Topic Modeling


**Latent Dirichlet Allocation (LDA)** is a probabilistic model that treats each document as a "mixture" of multiple topics and each topic as a "mixture" of words. While NMF produces distinct, categorical assignments, LDA is designed to capture the nuance of overlapping themes.

**Key characteristics of LDA:**
* **Probabilistic logic:** It assumes students may be talking about multiple things at once (e.g., 60% about workload and 40% about peer support).
* **Input requirements:** Unlike NMF, LDA performs best with **raw count vectors** (Bag-of-Words) rather than TF-IDF.
* **Flexibility:** It is often better suited for longer, more complex responses where themes are naturally intertwined.

In the code below, we switch to our count-vectorized matrix and apply the LDA algorithm to discover the same number of topics for comparison.

<br>

> **Instructor Perspective:** If NMF is about finding the "primary" bucket for a comment, LDA is about understanding the "ingredients" that make up that comment. For IR reporting, NMF often provides "crisper" results, but LDA can reveal deeper, more complex relationships in student feedback.

In [ ]:
count_vec = CountVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2), min_df=2)
count_matrix = count_vec.fit_transform(Free_Response_training19['Comment'])
feature_names_count = count_vec.get_feature_names_out()

lda_model = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=20)
W_lda = lda_model.fit_transform(count_matrix)

print("LDA Topics (top 8 words each):")
print_top_words(lda_model.components_, feature_names_count, label='Topic')


LDA Topics (top 8 words each):
  Topic 0: work, feel, feedback, pushes, best, environment, best work, pushes best
  Topic 1: campus, classes, mixed, mixed experience, experience, experience classes, classes engaging, engaging
  Topic 2: study, study groups, wish, wish peer, manage, manage okay, groups, peer
  Topic 3: academic, support, hard, support adequate, academic support, courses hard, adequate, courses


In [ ]:
topic_labels_lda = {
    0: "Performance & Feedback",
    1: "Campus & Class Experience",
    2: "Study Habits & Peer Support",
    3: "Academic Support & Course Rigor"
}

print("LDA Topic Labels:")
for t_id, label in topic_labels_lda.items():
    print(f"Topic {t_id}: {label}")

LDA Topic Labels:
Topic 0: Performance & Feedback
Topic 1: Campus & Class Experience
Topic 2: Study Habits & Peer Support
Topic 3: Academic Support & Course Rigor



## 6. Assigning Dominant Topics


Once the model has been trained, we have a **Document-Topic Matrix (W)**. Each row in this matrix represents a student's response, and each column represents a topic. The values are weights indicating how strongly a response aligns with a particular theme.

To make this data actionable for Institutional Research, we identify the **highest-weight topic** for each row and assign it as that student's **Dominant Topic**. This process converts complex mathematical weights into a simple categorical variable that can be used for:
* **Segmenting survey results** by primary student concern.
* **Filtering longitudinal data** to see how themes shift over time.
* **Merging with demographic data** to identify if certain student groups are discussing specific issues more frequently.

In the code below, we use `np.argmax` to find the strongest topic for each student and add these new features directly to our dataframe for comparison.

<br>

> **Instructor Perspective:** Assigning a dominant topic is the "classification" step of unsupervised learning. While a student might touch on multiple themes, identifying the most prominent one allows us to create the clean, high-level visualizations that stakeholders expect.

In [ ]:
# Add NMF and LDA results to our dataframe
df_Topic_Modeling = ML_Survey_Data19[['SID','Free_Response_Text']]
df_Topic_Modeling['Dominant_Topic_NMF'] = np.argmax(W_nmf, axis=1)
df_Topic_Modeling['Dominant_Topic_LDA'] = np.argmax(W_lda, axis=1)

# Visualize NMF topic distribution
topic_counts = (df_Topic_Modeling['Dominant_Topic_NMF']
                .value_counts().sort_index().reset_index())
topic_counts.columns = ['Topic', 'Count']

fig = px.bar(topic_counts, x='Topic', y='Count',
             title='Dominant Topic Distribution — NMF',
             labels={'Topic': 'Topic Number', 'Count': 'Number of Responses'})
fig.show()

df_Topic_Modeling


/tmp/ipython-input-3790229537.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipython-input-3790229537.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,SID,Free_Response_Text,Dominant_Topic_NMF,Dominant_Topic_LDA
0,UHKR1SFYP,"When multiple deadlines hit at once, I have to...",2,2
1,UHNF02H9O,"When deadlines pile up, I feel like I’m just t...",0,0
2,UHOFN46WE,"My academic journey has been very rewarding, a...",0,0
3,UHP7B9BTV,Sometimes I don’t understand what instructors ...,0,0
4,UI30HVLIL,"I’ve been feeling overwhelmed, and I’m trying ...",2,0
...,...,...,...,...
5157,N26B5Y6HE,"I know support exists, but it can be hard to r...",2,0
5158,N28LHIN72,"The workload can feel heavy at times, but I ca...",1,1
5159,N299IIGIN,I am confident in my study habits and ability ...,2,0
5160,N2C5YLPI1,I am adapting to college life and finding my r...,0,3


## 7. Reporting for IR Stakeholders



Topic models are most useful when translated into institutional language. A practical reporting pattern:

1. **Name each topic** based on its top terms
2. **Count** how many responses fall in each topic (and what % of total)
3. **Show example comments** so readers understand what each topic really means


In [ ]:
def sample_comments(df, topic_col, topic_id, n=3, seed=42):
    subset = df[df[topic_col] == topic_id]
    return subset.sample(min(n, len(subset)), random_state=seed)['Free_Response_Text'].tolist()

total = len(df_Topic_Modeling)
print("=" * 60)
print("NMF TOPIC SUMMARY REPORT — Fall 2019 Cohort")
print("=" * 60)
for t_id, label in topic_labels_nmf.items():
    count = (df_Topic_Modeling['Dominant_Topic_NMF'] == t_id).sum()
    pct = 100 * count / total
    examples = sample_comments(df_Topic_Modeling, 'Dominant_Topic_NMF', t_id)
    print(f"\nTopic {t_id}: {label}  ({count} responses, {pct:.1f}%)")
    for ex in examples:
        print(f"  • {ex}")


NMF TOPIC SUMMARY REPORT — Fall 2019 Cohort

Topic 0: Academic Expectations & Learning Experience  (1648 responses, 31.9%)
  • The transition to college has been tougher than I expected, but I'm determined to grow.
  • I'm finding some aspects of college challenging and am seeking resources to improve.
  • Some classes feel more challenging than I expected, and I’m learning how to meet those expectations.

Topic 1: Workload Management & Stress  (887 responses, 17.2%)
  • I feel prepared in some areas, but there are topics where I need more practice or review.
  • I feel prepared in some areas, but there are topics where I need more practice or review.
  • I hope to find my academic footing soon and start seeing better results.

Topic 2: Support, Feedback & Improvement Strategies  (2020 responses, 39.1%)
  • I want to use more campus resources, but scheduling and time constraints get in the way.
  • I’m still figuring out the best way to study, and talking with classmates helps me stay 

In [ ]:
def sample_comments(df, topic_col, topic_id, n=3, seed=42):
    subset = df[df[topic_col] == topic_id]
    return subset.sample(min(n, len(subset)), random_state=seed)['Free_Response_Text'].tolist()

total = len(df_Topic_Modeling)
print("=" * 60)
print("LDA TOPIC SUMMARY REPORT — Fall 2019 Cohort")
print("=" * 60)
for t_id, label in topic_labels_lda.items():
    count = (df_Topic_Modeling['Dominant_Topic_LDA'] == t_id).sum()
    pct = 100 * count / total
    examples = sample_comments(df_Topic_Modeling, 'Dominant_Topic_LDA', t_id)
    print(f"\nTopic {t_id}: {label}  ({count} responses, {pct:.1f}%)")
    for ex in examples:
        print(f"  • {ex}")


LDA TOPIC SUMMARY REPORT — Fall 2019 Cohort

Topic 0: Performance & Feedback  (1295 responses, 25.1%)
  • I sometimes get feedback that I don’t fully understand, and I need help turning it into concrete next steps.
  • I often start assignments too late, and then I’m rushing to finish instead of learning the material.
  • I’ve been feeling overwhelmed, and I’m trying to figure out where to get help before I fall further behind.

Topic 1: Campus & Class Experience  (1581 responses, 30.6%)
  • Rubrics and comments are helpful for fine-tuning my work, especially on larger assignments.
  • I tend to do well on exams and projects, and I appreciate feedback that pushes me to refine my thinking.
  • I feel academically prepared, so I’m comfortable taking on rigorous coursework.

Topic 2: Study Habits & Peer Support  (1028 responses, 19.9%)
  • Sometimes feedback helps a lot, but other times I’m not sure how to apply it on the next assignment.
  • When multiple deadlines hit at once, I have to


## 8. Wrap-Up


In this module, we moved beyond individual word frequencies to discover the thematic structures hidden within student comments. By applying unsupervised learning, you have gained the ability to categorize large-scale qualitative data into actionable institutional insights.

**What you accomplished:**
* **Applied NMF (on TF-IDF) and LDA (on count vectors)** to discover recurring themes in student voices.
* **Assigned dominant topics** to individual responses, creating new categorical features for analysis.
* **Produced stakeholder-ready reports** that bridge the gap between algorithmic output and institutional action.

<br>

#### Key Decision Points for Practice:
* **Number of Topics:** Start with a small range (3–6). Too many topics make interpretation difficult, while too few may blend distinct student concerns into a single "catch-all" category.
* **Algorithm Choice:** Remember that **NMF** often provides "crisper," more distinct topics for short survey text, while **LDA** is better for capturing overlapping themes in longer documents.
* **Human Validation:** Always review the top-weighted words and sample comments before finalized a topic label. The model finds the clusters, but the analyst provides the meaning.

> **Instructor Perspective:** Topic modeling is a powerful starting point, but it is not the end of the analysis. As you move forward, consider how these discovered topics correlate with quantitative outcomes like retention, GPA, or satisfaction scores.

<br>


**Coming up next:** In **7.5**, we will build on this same comment data to estimate the **emotional tone** of student voices using Sentiment Analysis.